In [1]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from toolhouse import Toolhouse
from dotenv import load_dotenv
import os
from nomic import embed
from supabase import create_client, Client
from openai import OpenAI
from prisma import Prisma
from groq import Groq
from fastapi.middleware.cors import CORSMiddleware
from fastapi.encoders import jsonable_encoder
import time
import logging
import traceback
logging.basicConfig(level=logging.DEBUG)



In [2]:
load_dotenv()

#logging.basicConfig(level=logging.DEBUG)

# Initialize OpenAI client with your API key
# Create the Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL = 'llama3-groq-70b-8192-tool-use-preview'
#MODEL = 'gpt-4o-mini'
#client = OpenAI()
# Create the tools
th = Toolhouse()
th.set_metadata('timezone', -4)

#MODEL = "llama3-groq-70b-8192-tool-use-preview"

DEBUG:httpx:load_ssl_context verify=True cert=None trust_env=True http2=False
DEBUG:httpx:load_verify_locations cafile='/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/certifi/cacert.pem'


In [3]:
async def getMessages(userId, conversationId):
    db = Prisma ()
    await db.connect()
    messages = []
    conversation = await db.conversation.find_first_or_raise(
        where={
            'id': conversationId,
            'userId': userId,
        },
        include={
            'messages': True
        }
    )
    for msg in conversation.messages:
        messages.append({"role": msg.role, "content": msg.content})
    await db.disconnect()
    return messages

In [4]:
async def updateMessages(newMessage, conversationId):
    db = Prisma ()
    await db.connect()
    conversation = await db.conversation.update(
        where={
            'id': conversationId
        },
        data={
            'messages':{
                'create': newMessage
            }
        }
    )
    await db.disconnect()
    return conversation

In [5]:
async def retrieveDocs(query, userId):
    queryVector = embed.text(
        texts = [query],
        model='nomic-embed-text-v1.5',
        task_type='search_query',     
    )
    queryEmbedding = queryVector['embeddings'][0]
    #print(queryEmbedding)
    embedding_array = f"{queryEmbedding}"

    db = Prisma()
    await db.connect()
    try:
        docIds = await db.query_raw(
           '''
            SELECT "documentId"
            FROM "DocumentChunks"
            WHERE "userId" = CAST($1 AS TEXT)
            ORDER BY "embedding" <-> CAST($2 AS vector) ASC
            LIMIT 5
            ''',
            userId,
            embedding_array 
        )
        
        return docIds
            
    finally:
        await db.disconnect()

In [6]:
async def getDocContext(docIds):
    db = Prisma()
    await db.connect()
    context = ""
    for docId in docIds:
        doc = await db.document.find_first_or_raise(
            where={
                'id': docId['documentId']
            },
        )
        context += doc.content
    await db.disconnect()
    if (context != ""):
        context = "\n\n The following is the relevant user's journal contents: " + context
    return context

In [7]:
# Initialize FastAPI client
app = FastAPI()

origins = [
    "http://localhost:3000",
]

app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [8]:
class Message (BaseModel):
    role: str
    content: str
    
# Function to handle one step of the conversation
async def chat_with_assistant(input_str, userId, conversationId):
    start_time = time.time()
    # Get the documents and add them to the context
    docIds = await retrieveDocs(input_str, userId)
    print("Embedding query and retrieving docIds: " + str(time.time()-start_time))
    userDocContext = await getDocContext(docIds)
    messages = await getMessages(userId, conversationId)
    print("Retrieving doc contents: " + str(time.time()-start_time))
    
    # Append the user input to the chat history
    promptFormated = [{
        "role": "user",
        "content": "This is the user's key query: " + input_str + "| end of query \n\n" + userDocContext
    }]
    messages += promptFormated
    print(messages)
    # Create the response
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=th.get_tools(),
        temperature=1.2
    )
    print("Initial feed??: " + str(time.time()-start_time))

    tool_run = th.run_tools(response)
    print(f"Type of tool_run: {type(tool_run)}")

    promptFormated.extend(tool_run)
    print("With tools: " + str(time.time()-start_time))

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages + promptFormated,
        tools=th.get_tools(),
        temperature=1.2
    )
    # Append the response to the chat history
    newMessage = {
        "role": "assistant",
        "content": response.choices[0].message.content
    }
    print("Final: " + str(time.time()-start_time))
    #print(messages)
    # Print the assistant's response
    await updateMessages(newMessage, conversationId)
    return (Message(role="assistant", content = response.choices[0].message.content))


In [9]:
class ChatRequest (BaseModel):
    conversationId: str
    title: str
    userId: str
    userMessage: str
    

@app.post("/chatbot/chat", response_model=Message)  # This line decorates 'translate' as a POST endpoint
async def chat(request: ChatRequest):
    try:
        # Call your translation function
        message = await chat_with_assistant(request.userMessage, request.userId, request.conversationId)
        return jsonable_encoder(message)
    except Exception as e:
        # Handle exceptions or errors during translation
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000, log_level="debug")


INFO:     Started server process [33354]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:57831 - "OPTIONS /chatbot/chat HTTP/1.1" 200 OK


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api-atlas.nomic.ai:443
DEBUG:urllib3.connectionpool:https://api-atlas.nomic.ai:443 "GET /v1/user HTTP/11" 307 0
DEBUG:urllib3.connectionpool:https://api-atlas.nomic.ai:443 "GET /v1/user/ HTTP/11" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api-atlas.nomic.ai:443
DEBUG:urllib3.connectionpool:https://api-atlas.nomic.ai:443 "POST /v1/embedding/text HTTP/11" 200 None
DEBUG:prisma._base_client:datasources: None
DEBUG:prisma.engine._query:Connecting to query engine
DEBUG:prisma.engine.utils:Expecting local query engine /Users/henry/maybility/maybility-ai/prisma-query-engine-darwin
DEBUG:prisma.engine.utils:Expecting global query engine /Users/henry/.cache/prisma-python/binaries/5.17.0/393aa359c9ad4a4bb28630fb5613f9c281cde053/prisma-query-engine-darwin
DEBUG:prisma.engine.utils:Query engine found from the Prisma CLI generated path: /Users/henry/.cache/prisma-python/binaries/5.17.0/393aa359c9ad4a4bb286

Embedding query and retrieving docIds: 2.8684608936309814


DEBUG:prisma.engine._http:Constructed GET request to http://localhost:57870/status
DEBUG:prisma.engine._http:Request headers: {'Accept': 'application/json'}
DEBUG:prisma.engine._http:Request content: None
DEBUG:httpcore.connection:connect_tcp.started host='localhost' port=57870 local_address=None timeout=30 socket_options=None
DEBUG:httpcore.connection:connect_tcp.failed exception=ConnectError(OSError('All connection attempts failed'))
DEBUG:prisma.engine._query:Could not connect to query engine due to All connection attempts failed; retrying...
DEBUG:prisma.engine._http:Constructed GET request to http://localhost:57870/status
DEBUG:prisma.engine._http:Request headers: {'Accept': 'application/json'}
DEBUG:prisma.engine._http:Request content: None
DEBUG:httpcore.connection:connect_tcp.started host='localhost' port=57870 local_address=None timeout=30 socket_options=None
DEBUG:httpcore.connection:connect_tcp.failed exception=ConnectError(OSError('All connection attempts failed'))
DEBUG:pr

...

DEBUG:urllib3.connectionpool:https://api.toolhouse.ai:443 "POST /v1/get_tools HTTP/11" 200 525
DEBUG:groq._base_client:Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'json_data': {'messages': [{'role': 'user', 'content': 'Nothing much really I am kind of busy unfortunately, I will be working on this project though'}, {'role': 'assistant', 'content': "Based on your journal, it seems like you have several projects and long-term focuses to work on. Considering you mentioned being busy, I would recommend prioritizing one of your key projects or long-term focuses for today. Since you mentioned you will be working on a project, let's focus on your Project Maybility. You could start by implementing the RAG benchmarks or working on indexing document chunks for faster RAG. These tasks could help you make progress on your project and potentially alleviate some of your busyness."}, {'role': 'assistant', 'content': 'Hi how may I help you today?'}, {'role':

Initial feed??: 14.266234159469604
Type of tool_run: <class 'list'>
With tools: 14.266324996948242


DEBUG:urllib3.connectionpool:https://api.toolhouse.ai:443 "POST /v1/get_tools HTTP/11" 200 525
DEBUG:groq._base_client:Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'json_data': {'messages': [{'role': 'user', 'content': 'Nothing much really I am kind of busy unfortunately, I will be working on this project though'}, {'role': 'assistant', 'content': "Based on your journal, it seems like you have several projects and long-term focuses to work on. Considering you mentioned being busy, I would recommend prioritizing one of your key projects or long-term focuses for today. Since you mentioned you will be working on a project, let's focus on your Project Maybility. You could start by implementing the RAG benchmarks or working on indexing document chunks for faster RAG. These tasks could help you make progress on your project and potentially alleviate some of your busyness."}, {'role': 'assistant', 'content': 'Hi how may I help you today?'}, {'role':

Final: 15.340185165405273


DEBUG:prisma.engine._http:Constructed GET request to http://localhost:50693/status
DEBUG:prisma.engine._http:Request headers: {'Accept': 'application/json'}
DEBUG:prisma.engine._http:Request content: None
DEBUG:httpcore.connection:connect_tcp.started host='localhost' port=50693 local_address=None timeout=30 socket_options=None
DEBUG:httpcore.connection:connect_tcp.failed exception=ConnectError(OSError('All connection attempts failed'))
DEBUG:prisma.engine._query:Could not connect to query engine due to All connection attempts failed; retrying...
DEBUG:prisma.engine._http:Constructed GET request to http://localhost:50693/status
DEBUG:prisma.engine._http:Request headers: {'Accept': 'application/json'}
DEBUG:prisma.engine._http:Request content: None
DEBUG:httpcore.connection:connect_tcp.started host='localhost' port=50693 local_address=None timeout=30 socket_options=None
DEBUG:httpcore.connection:connect_tcp.failed exception=ConnectError(OSError('All connection attempts failed'))
DEBUG:pr